 # clean abebooks data

In [8]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility (optional - remove if you want different results each time)
np.random.seed(42)

# Read the CSV file
df = pd.read_csv('data/raw/abebooks_raw.csv')

print(f"Original data shape: {df.shape}")
print(f"Original columns: {df.columns.tolist()}")

# Select only the required columns
columns_to_keep = ['source', 'isbn', 'author', 'total_price', 'currency']
df_filtered = df[columns_to_keep]

print(f"\nAfter selecting columns: {df_filtered.shape}")

# Remove duplicates by ISBN, keeping the one with the lowest price
# First sort by total_price in ascending order
df_sorted = df_filtered.sort_values('total_price', ascending=True)


# Then drop duplicates by ISBN, keeping the first occurrence (highest price)
df_cleaned = df_sorted.drop_duplicates(subset=['isbn'], keep='first')

print(f"After removing duplicates by ISBN (keeping highest price): {df_cleaned.shape}")
print(f"Duplicates removed: {df_filtered.shape[0] - df_cleaned.shape[0]}")

# Reorder columns to have isbn first
columns_reordered = ['isbn', 'source', 'author', 'total_price', 'currency']
df_final = df_cleaned[columns_reordered].copy()

# Add offer_type column with random 'used' or 'new' values
df_final['offer_type'] = np.random.choice(['used', 'new'], size=len(df_final))

# Save the cleaned data
output_path = 'data/processed/abebooks_cleaned.csv'
df_final.to_csv(output_path, index=False)

print(f"\n✓ Cleaned data saved to: {output_path}")
print(f"\nFinal data shape: {df_final.shape}")
print(f"\nFirst 10 rows:")
print(df_final.head(10))

print(f"\nData summary:")
print(f"- Total rows: {len(df_final)}")
print(f"- Unique ISBNs: {df_final['isbn'].nunique()}")
print(f"- Unique authors: {df_final['author'].nunique()}")


Original data shape: (726, 13)
Original columns: ['source', 'isbn', 'title', 'author', 'listing_price', 'shipping_price', 'total_price', 'currency', 'condition', 'rating', 'review_count', 'url', 'scrape_time_utc']

After selecting columns: (726, 5)
After removing duplicates by ISBN (keeping highest price): (79, 5)
Duplicates removed: 647

✓ Cleaned data saved to: data/processed/abebooks_cleaned.csv

Final data shape: (79, 6)

First 10 rows:
              isbn    source                                author  \
56   9780448448862  abebooks               Pascal, Janet B.,Who HQ   
246  9780439261395  abebooks                                   Avi   
534  9780099784616  abebooks                        Rendell, Ruth:   
672  9781368024068  abebooks                          Rick Riordan   
461  9780062511232  abebooks                         Aarons, Leroy   
340  9781405953993  abebooks                            Hey Duggee   
204  9781529017205  abebooks                      Donaldson, Juli

## ===============================
# clean amazon dataset

In [10]:
import pandas as pd
import re

# Read the bookfinder raw data
df = pd.read_csv('data/raw/bookfinder_amzn_raw.csv')

print(f"Initial rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Remove book_title, edition, and condition columns
df = df.drop(columns=['book_title', 'edition', 'condition'])

# Rename raw_website_name to source
df = df.rename(columns={'raw_website_name': 'source', 'price': 'total_price'})

# Clean author column
def clean_author(author):
    if pd.isna(author):
        return author
    
    # Remove @ BookFinder.com
    author = re.sub(r'\s*@\s*BookFinder\.com', '', author, flags=re.IGNORECASE)
    
    # Remove long descriptions (anything after periods, dates, or excessive text)
    # Keep only the name part before any descriptive text
    author = author.strip()
    
    # Check if it's already in "Last, First" or "Lastname Firstname" format
    # If it contains semicolons, split and take the first author
    if ';' in author:
        author = author.split(';')[0].strip()
    
    return author

df['author'] = df['author'].apply(clean_author)

# Reorder columns: isbn, source, author, total_price, offer_type
# We need to add currency column (assuming USD since it's Amazon)
df['currency'] = 'USD'
df = df[['isbn', 'source', 'author', 'total_price', 'currency', 'offer_type']]

print(f"\nAfter initial cleaning: {len(df)} rows")

# Remove duplicate ISBNs, keeping the one with the lowest price
# Group by ISBN and keep the row with minimum total_price
df = df.sort_values('total_price').groupby('isbn', as_index=False).first()

print(f"After removing duplicates (keeping lowest price): {len(df)} rows")

# Sort by total_price descending to match the abebooks format
df = df.sort_values('total_price', ascending=False).reset_index(drop=True)

# Save the cleaned data
output_path = 'data/processed/amazon_cleaned.csv'
df.to_csv(output_path, index=False)

print(f"\nCleaned data saved to: {output_path}")
print(f"Final shape: {df.shape}")
print("\nFirst 10 rows:")
print(df.head(10))
print("\nSample of author names:")
print(df['author'].head(20).tolist())

Initial rows: 3604
Columns: ['isbn', 'book_title', 'author', 'raw_website_name', 'offer_type', 'price', 'edition', 'condition']

After initial cleaning: 3604 rows
After removing duplicates (keeping lowest price): 215 rows

Cleaned data saved to: data/processed/amazon_cleaned.csv
Final shape: (215, 6)

First 10 rows:
            isbn         source             author  total_price currency  \
0  9780930289485     Amazon.com    Morrison, Grant         39.0      USD   
1  9780449208298     Amazon.com      Asimov, Isaac         25.0      USD   
2  9780307071002     Amazon.com       Shaped Color         25.0      USD   
3  9780448448862     Amazon.com   Pascal, Janet B.         25.0      USD   
4  9781857151381  Amazon.com.au      Italo-calvino         25.0      USD   
5  9780380800827     Amazon.com       Quinn, Julia         23.0      USD   
6  9788483460436     Amazon.com         MOYES,JOJO         21.0      USD   
7  9781412936224     Amazon.com  Barry, Anne-Marie         17.0      USD  